## Changelog
- parent: 20260508_101229_6d2882b7
- change: replace the equal-weight blend (mean of expm1 predictions per model) with a Ridge(alpha=1, positive=True) meta-learner trained on out-of-fold predictions. Pipelines and per-model hyperparameters are unchanged. Reports per-model OOF CV, equal-weight OOF CV, and stacked OOF CV side-by-side so the stacking lift is measured directly.
- hypothesis: the equal-weight blend gives XGBoost (CV ~0.115) the same vote as Lasso (CV ~0.111). A meta-learner with non-negative weights should down-weight XGBoost relative to Lasso/KRR and pick up some of the 30–100 LB the recap estimated. Positive-constrained because negative weights overfit on 1455 rows; alpha=1 because the design matrix is 5×4 in OOF space and over-fitting the meta is the main risk.


In [ ]:
import sys
import numpy as np
import pandas as pd

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

    if str(comp_dir) not in sys.path:
        sys.path.insert(0, str(comp_dir))

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")


In [ ]:
# see eda-TotalSF.ipynb — drop the mega-house outliers (TotalSF > 7000)
_outlier_mask = (
    train_data_raw["TotalBsmtSF"]
    + train_data_raw["1stFlrSF"]
    + train_data_raw["2ndFlrSF"]
) > 7000
train_data = train_data_raw.loc[~_outlier_mask].reset_index(drop=True)

DROP = ["Id", "SalePrice"]
X      = train_data.drop(columns=DROP, errors="ignore").copy()
X_test = test_data_raw.drop(columns=DROP, errors="ignore").copy()
y      = np.log1p(train_data["SalePrice"])

# MSSubClass is a nominal int code — cast to string so the encoder treats it
# as a category rather than an ordered number.
X["MSSubClass"]      = X["MSSubClass"].astype(str)
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

NUMERIC     = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = {len(X.columns)} total")


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class CategoryCaster(BaseEstimator, TransformerMixin):
    """Pin categorical vocabulary at fit-time, replay at transform."""

    def fit(self, X, y=None):
        self.vocab_ = {}
        cat_like = X.select_dtypes(include=["object", "string", "category"]).columns
        for c in cat_like:
            self.vocab_[c] = sorted(X[c].dropna().astype(str).unique())
        return self

    def transform(self, X):
        out = X.copy()
        for c, cats in self.vocab_.items():
            if c in out.columns:
                out[c] = pd.Categorical(out[c].astype(object), categories=cats)
        return out


class BoxCoxSkewed(BaseEstimator, TransformerMixin):
    """boxcox1p(lam) on numeric columns with |skew| > threshold (per-fold)."""

    def __init__(self, threshold: float = 0.75, lam: float = 0.15):
        self.threshold = threshold
        self.lam = lam

    def fit(self, X, y=None):
        from scipy.stats import skew
        numeric = X.select_dtypes(include="number").columns
        skews = X[numeric].apply(lambda s: skew(s.dropna()))
        self.skewed_cols_ = skews[skews.abs() > self.threshold].index.tolist()
        return self

    def transform(self, X):
        from scipy.special import boxcox1p
        out = X.copy()
        for c in self.skewed_cols_:
            if c in out.columns:
                out[c] = boxcox1p(out[c], self.lam)
        return out


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler
from sklearn.linear_model import Lasso, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import KFold

from xgboost import XGBRegressor

from utils.ames_sklearn_pipeline import AmesNAImputer, AmesEncoder
from utils.ames_feature_engineering import (
    add_size_features, add_temporal_features, add_bath_features,
    add_quality_size_features,
)

xgb_pipe = Pipeline([
    ("na",            AmesNAImputer()),
    ("quality_size",  FunctionTransformer(
                          add_quality_size_features,
                          kw_args={"drop_originals": False},
                      )),
    ("size",          FunctionTransformer(
                          add_size_features,
                          kw_args={"drop_originals": True},
                      )),
    ("fe",            FunctionTransformer(
                          add_temporal_features,
                          kw_args={"drop_originals": True},
                      )),
    ("bath",          FunctionTransformer(
                          add_bath_features,
                          kw_args={"drop_originals": False},
                      )),
    ("cat_cast",      CategoryCaster()),
    ("model",         XGBRegressor(
                          tree_method="hist",
                          enable_categorical=True,
                          learning_rate=0.05,
                          n_estimators=600,
                          max_depth=4,
                          min_child_weight=1,
                          reg_lambda=1,
                          subsample=0.8,
                          colsample_bytree=0.8,
                          random_state=42,
                          n_jobs=1,
                          verbosity=0,
                      )),
])

def make_linear_pipe(model):
    return Pipeline([
        ("na",            AmesNAImputer()),
        ("quality_size",  FunctionTransformer(
                              add_quality_size_features,
                              kw_args={"drop_originals": False},
                          )),
        ("size",          FunctionTransformer(
                              add_size_features,
                              kw_args={"drop_originals": True},
                          )),
        ("fe",            FunctionTransformer(
                              add_temporal_features,
                              kw_args={"drop_originals": True},
                          )),
        ("bath",          FunctionTransformer(
                              add_bath_features,
                              kw_args={"drop_originals": False},
                          )),
        ("skew",      BoxCoxSkewed(threshold=0.75, lam=0.15)),
        ("encoder",   AmesEncoder()),
        ("scaler",    RobustScaler()),
        ("model",     model),
    ])

lasso_pipe = make_linear_pipe(Lasso(alpha=0.0005, random_state=1, max_iter=10000))
ridge_pipe = make_linear_pipe(Ridge(alpha=10, random_state=2))
krr_pipe   = make_linear_pipe(KernelRidge(alpha=0.6, kernel="polynomial", degree=2, coef0=2.5))

BASE_PIPES = {
    "XGBoost": xgb_pipe,
    "Lasso": lasso_pipe,
    "Ridge": ridge_pipe,
    "KRR": krr_pipe,
}


In [ ]:
from sklearn.base import clone

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# OOF predictions per base model (in log-price space, same as y)
def oof_predict(pipe, X, y, cv):
    oof = np.zeros(len(X))
    for tr_idx, vl_idx in cv.split(X):
        p = clone(pipe)
        p.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        oof[vl_idx] = p.predict(X.iloc[vl_idx])
    return oof


print("Generating OOF predictions per base model...")
oof = {}
for name, pipe in BASE_PIPES.items():
    oof[name] = oof_predict(pipe, X, y, cv)
    rmse = np.sqrt(np.mean((oof[name] - y.values) ** 2))
    print(f"  {name:8s}: CV RMSE (log) = {rmse:.4f}")


# ---- Equal-weight blend (baseline reference)
oof_matrix = np.column_stack([oof[n] for n in BASE_PIPES])
eq_blend_oof = oof_matrix.mean(axis=1)
eq_rmse = np.sqrt(np.mean((eq_blend_oof - y.values) ** 2))
print(f"\nEqual-weight blend OOF RMSE (log): {eq_rmse:.4f}")


# ---- Ridge meta-learner on OOF predictions
from sklearn.linear_model import Ridge as _MetaRidge

meta = _MetaRidge(alpha=1.0, fit_intercept=True, positive=True)
meta.fit(oof_matrix, y)
print("\nMeta Ridge weights (positive-constrained):")
for n, w in zip(BASE_PIPES, meta.coef_):
    print(f"  {n:8s}: {w:.3f}")
print(f"  intercept = {meta.intercept_:.3f}")

stacked_oof = meta.predict(oof_matrix)
stacked_rmse = np.sqrt(np.mean((stacked_oof - y.values) ** 2))
print(f"\nStacked (Ridge meta) OOF RMSE (log): {stacked_rmse:.4f}")
print(f"Lift vs equal-weight: {eq_rmse - stacked_rmse:+.4f} (lower is better)")


# ---- Refit each base on full data, then meta-predict
for pipe in BASE_PIPES.values():
    pipe.fit(X, y)


In [ ]:
test_logs = np.column_stack([
    pipe.predict(X_test) for pipe in BASE_PIPES.values()
])
test_log = meta.predict(test_logs)
test_pred = np.expm1(test_log)

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)
print("submission.csv written.")
